# SewerTris Repository Documentation

**SewerTris** is a synthetic sanitary sewer system generator and EPA-SWMM benchmarking framework. This notebook is intended to become the living documentation spine for the repository: a readable explanation of the model philosophy, installation, workflow, project files, sibling ensembles, and plotting utilities.

The key idea is simple: SewerTris creates physically plausible digital sewer systems that can be regenerated, perturbed, cloned, and compared under controlled experimental conditions. The package can be used step by step with individual functions, or through the `SewerTrisProject` interface, which records parameters and artifacts in a reproducible project JSON.

## Table of Contents

1. Model philosophy
2. Installation
3. Repository structure
4. Two ways to use SewerTris
5. The twelve-step modeling workflow
6. Project outputs and JSON metadata
7. Digital siblings and ensembles
8. Flow-component experiments
9. Plotting and diagnostics
10. Recommended example notebooks
11. Documentation maintenance checklist

## 1. Model Philosophy

SewerTris treats synthetic sewer networks as **digital experimental systems**. Instead of relying on one observed network, the user can generate many structurally plausible systems and ask controlled questions about design, monitoring, rainfall response, infiltration, or hydraulic behavior.

The model philosophy has four pillars:

| Pillar | Meaning in SewerTris |
|---|---|
| Controlled stochasticity | Urban layout, land use, rainfall, and inflow fields can vary through explicit seeds and parameter changes. |
| Physical coherence | Roads, DEMs, manholes, pipe slopes, pipe diameters, and SWMM inputs are generated with consistent spatial and hydraulic assumptions. |
| Reproducibility | The project API records step parameters, output paths, and scenario metadata in `sewertris_project.json`. |
| Sibling analysis | A base model can be cloned into many related variants, or siblings, so that ensemble differences are tied to explicit changes. |

A useful way to think about SewerTris is not as a single model generator, but as an **experiment factory**. A base project defines one internally consistent synthetic sewer system. Siblings then perturb pieces of that base project while preserving lineage and rerun provenance.

## 2. Installation

Clone the repository and create the conda environment:

```bash
git clone https://github.com/Perez-HydroSystems/sewertris.git
cd sewertris
conda env create -f environment_sewertris.yml
conda activate sewertris
```

For development, install the package in editable mode from the repository root:

```bash
pip install -e .
```

Run the tests:

```bash
pytest
```

Important runtime dependencies include `geopandas`, `rasterio`, `shapely`, `networkx`, `xarray`, `matplotlib`, `pyswmm`, and `swmm-toolkit`. SWMM simulations require the PySWMM stack to be correctly installed in the active environment.

In [3]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src" / "sewertris").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

EXAMPLES_DIR = PROJECT_ROOT / "Examples"
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import sewertris as st

print("SewerTris version:", st.__version__)
print("Project API available:", hasattr(st, "SewerTrisProject"))
print("Ensemble plotting available:", hasattr(st, "plot_ensemble_results"))

SewerTris version: 0.1.0
Project API available: True
Ensemble plotting available: True


## 3. Repository Structure

The current repository is organized around a Python package, example notebooks, tests, and documentation assets.

```text
sewertris/
|-- src/sewertris/
|   |-- project.py        # SewerTrisProject, scenarios, project JSON, siblings
|   |-- ensemble.py       # Subprocess-friendly sibling and simulation workers
|   |-- layout.py         # Tetromino layouts and block exports
|   |-- roads.py          # Road extraction from generated blocks
|   |-- topography.py     # Synthetic DEM generation and DEM embedding
|   |-- sewer_network.py  # Manholes, main/secondary/tertiary pipe routing
|   |-- hydrology.py      # Baseflow, GWI, RDII, rainfall and rasters
|   |-- design.py         # Pipe sizing, slopes, inverts, DWF patterns
|   |-- swmm.py           # SWMM input creation, simulation, flow decomposition
|   |-- plots.py          # Spatial, project comparison, and ensemble plots
|-- examples/             # Step-by-step, project, sibling, and ensemble notebooks
|-- tests/                # Package and workflow smoke tests
|-- docs/                 # Documentation notebooks and images
|-- pyproject.toml        # Package metadata and dependencies
|-- environment_sewertris.yml
```

The `src/sewertris/project.py` module is the orchestration layer. It does not replace the lower-level functions; it records their use and makes the workflow reproducible.

## 4. Two Ways to Use SewerTris

SewerTris intentionally supports two usage styles.

### A. Step-by-step function API

Use this when developing algorithms, testing one module, or teaching the mechanics of a specific step. For example, you can call layout, road, hydrology, design, or SWMM functions directly from `sewertris.layout`, `sewertris.roads`, `sewertris.hydrology`, and related modules.

### B. Project API

Use `SewerTrisProject` when you want a complete reproducible model. The project API writes artifacts to a structured output directory, records step parameters, and creates a `sewertris_project.json` file that can be reloaded later.

The project API is strongly recommended for:

- reproducible examples,
- sibling generation,
- ensemble preparation,
- comparisons between base models,
- publishing datasets or benchmark experiments.

In [4]:
# Minimal project setup pattern. Set True to create a small documentation project folder.
RUN_PROJECT_SETUP_DEMO = False

from pathlib import Path
import sewertris as st

if RUN_PROJECT_SETUP_DEMO:
    project = st.SewerTrisProject(
        Path("Examples/output_documentation_demo"),
        cell_size_m=100,
        name="Documentation demo",
    )
    display(project.paths(as_str=True))

## 5. The Twelve-Step Modeling Workflow

The package follows a twelve-step workflow. Each step can be used directly through lower-level functions, but the project methods below are the recommended reproducible path.

| Step | Concept | Project method(s) | Core function families | Main outputs | Important variations |
|---:|---|---|---|---|---|
| 1 | Urban domain definition | `define_domain`, `build_domain_mask_from_shapefile` | `build_domain_mask_from_shapefile` | `domain_mask.npy`, `grid_meta.json` | Use a vector polygon or an existing raster-like mask. |
| 2 | Tetris block definition | `define_tetromino_set`, `define_tetrominoes` | `get_tetromino_set` | tetromino dictionary in project state | Full tetromino set or subsets such as `I_O_T_S_only`. |
| 3 | Stochastic Tetris completion | `complete_tetris_layout(seed=...)` | `fill_domain_with_tetrominoes_and_blocks`, block exports | `layout_blocks.gpkg` or shapefile | Reproducible layout seed; georeferenced or simple grid export. |
| 4 | Road network extraction | `generate_roads`, `extract_road_boundaries` | road boundary and centerline extraction | road centerlines, road polygons, outer shell | Road width and simplification tolerance. |
| 5 | Land-use assignment | `assign_land_use` | land-use distribution helpers | city blocks with land use | User-defined land-use probabilities and seed. |
| 6 | Synthetic DEM generation | `generate_topography` | `TopographyConfig`, DEM builders | `synthetic_dem.tif` | Outlet direction, elevation range, smoothing, drainage assumptions. |
| 7 | Sewer network generation | `generate_sewer_network_V2`, `embed_sewer_network_in_dem` | manhole extraction, main/secondary/tertiary routing | manholes, pipes, DEM with enforced slopes | V2 shortest-path tertiary routing is recommended; old method is retained for reference. |
| 8 | Sewer flow predesign | `predesign_flows` | baseflow, GWI, RDII accumulation | pipes with predesign flow fields | Land-use demand, GWI factor, RDII factor. |
| 9 | Pipe sizing and hydraulic properties | `design_pipes` | material, diameter, slopes, invert elevations | clean pipes and manholes | Standard diameters, roughness/material fractions, minimum slope/cover. |
| 10 | Dynamic SWMM input definition | `export_swmm`, `create_run`, scenario inflow methods | DWF, GWI, RDII, pollutant tags | SWMM `.inp` files | DWF patterns, pipe-length GWI, raster GWI, uniform RDII, raster RDII, generated rainfall. |
| 11 | EPA-SWMM simulation | `run_swmm` | `run_swmm_and_plot` | SWMM outputs | Monitored nodes and links. |
| 12 | Flow output decomposition | `decompose_flows` | `get_flow_components_from_node_pyswmm` | `flows.nc` | Outlet link/node selection; DWF/GWI/RDII tracer separation. |

In [5]:
# Full project workflow skeleton. This is documentation code; set RUN_WORKFLOW=True only
# after providing a domain path or domain mask and choosing output directories.
RUN_WORKFLOW = False

if RUN_WORKFLOW:
    project = st.SewerTrisProject("Examples/output_my_project", cell_size_m=100)

    # 1. Domain
    project.define_domain(shapefile_path="path/to/domain_boundary.shp", cell_size_m=100)

    # 2-3. Layout
    tetrominoes, colors = project.define_tetromino_set("full")
    project.complete_tetris_layout(georeferenced=True, seed=1000)

    # 4-6. Roads, land use, topography
    road_width = 10
    project.generate_roads(road_width=road_width, simplify_tol=0.5)
    project.extract_road_boundaries()
    project.assign_land_use(seed=42)
    project.generate_topography(config=st.TopographyConfig(outlet_direction="W"))

    # 7. Recommended network method: V2
    project.generate_sewer_network_V2(
        road_width=road_width,
        block_size=project.cell_size_m * 2,
        tertiary_block_size=project.cell_size_m * 10,
        tertiary_adverse_slope_weight=200.0,
    )
    project.embed_sewer_network_in_dem()

    # 8-9. Predesign and pipe design
    project.predesign_flows(land_use_info=None)
    project.design_pipes()

    # 10-12. SWMM scenario and flow decomposition
    project.export_swmm()
    scenario = project.create_run("bwf_gwi_rdii")
    scenario.assign_dwf_patterns()
    scenario.assign_gwi_from_pipe_length(coefficient=0.00001)
    scenario.add_subcatchment_rdii(timeseries=[["1/1/1990", "00:00", 0.0]])
    scenario.add_pollutant_tags()
    scenario.run_swmm(monitored_nodes=["OUTLET"], monitored_links=["P_OUTLET"])
    df = scenario.get_flow_components(link_id="P_OUTLET", node_id="OUTLET")
    scenario.save_flow_components(df)

    project.save()

## 6. Project Outputs and JSON Metadata

A `SewerTrisProject` output directory is more than a folder of files. It is a reproducible record of how the model was generated.

Typical outputs include:

```text
output_project/
|-- sewertris_project.json       # project metadata, lineage, step parameters, scenarios
|-- state/
|   |-- domain_mask.npy
|   |-- grid_meta.json
|-- layout_blocks.gpkg
|-- roads_centerlines.gpkg
|-- roads_polygons.gpkg
|-- synthetic_dem.tif
|-- manholes.gpkg
|-- pipes.gpkg
|-- sewer_model.inp
|-- flows.nc
|-- scenarios/
|   |-- bwf_gwi_rdii/
|       |-- sewer_model.inp
|       |-- flows.nc
```

The JSON stores:

- project name, output directory, vector format, and cell size,
- artifacts and scenario artifacts,
- ordered step records and parameters,
- sibling lineage and changes,
- scenario step records for SWMM variants.

This matters because siblings replay from the parent JSON. If a base project records V2 sewer network generation, siblings inherit V2 unless their `changes` explicitly override the network configuration.

In [6]:
# Useful project metadata inspection patterns
RUN_INSPECTION = False

if RUN_INSPECTION:
    project = st.SewerTrisProject.load("Examples/output_example_2_project/sewertris_project.json")
    print(project.project_file)
    print(project.artifacts().keys())
    print(project.step_parameters("03_stochastic_tetris_completion"))
    print(project.step_parameters("07_sewer_network_generation"))

## 7. Digital Siblings and Ensembles

A **sibling** is a cloned project that shares a parent model lineage but changes selected parameters. This is one of the core design features of SewerTris.

The sibling workflow has three parts:

1. Create a complete base project and save its `sewertris_project.json`.
2. Define a `changes` dictionary for each sibling.
3. Clone and rerun only the necessary steps from the parent parameters.

The project API infers the earliest step that must be rerun from the change keys. For example:

| Change key | Typical rerun start |
|---|---|
| `layout_seed`, `layout`, `seed` | stochastic layout completion |
| `road_width`, `roads` | road extraction |
| `land_use`, `land_use_distribution` | land-use assignment |
| `topography_config`, `dem` | DEM generation |
| `network`, `network_method`, `tertiary_*` | sewer network generation |
| `predesign`, `gwi_factor_ls_per_m`, `rdii_factor_ls_per_m2` | flow predesign |
| `pipe_design`, `materials`, `diameters` | pipe sizing |
| `rainfall`, `gwi`, `rdii`, `scenario`, `run_flow_components` | SWMM input/scenario stage |

The design intent is that a sibling change should be declarative: state what changed, and let the project rerun machinery determine how much of the workflow must be regenerated.

In [7]:
# Sibling specification examples
BASE_LAYOUT_SEED = 1000
BASE_RAINFALL_SEED = 42

def seed_only_changes(i):
    return {
        "layout_seed": BASE_LAYOUT_SEED + i,
        "rainfall": {
            "random_seed": BASE_RAINFALL_SEED + i,
        },
    }

def spatial_rdii_changes(i):
    return {
        "layout_seed": BASE_LAYOUT_SEED + i,
        "rainfall": {
            "random_seed": BASE_RAINFALL_SEED + i,
        },
        "rdii_raster": {
            "min_density": 0.0,
            "max_density": 5.0,
            "random_seed": 5000 + i,
        },
    }

def drainage_shift_changes(i):
    return {
        **seed_only_changes(i),
        "topography_config": {
            "outlet_direction": "W",
            "min_elevation": 280,
            "max_elevation": 290,
            "smoothing_factor": 1,
        },
    }

# Note: if the base project already used generate_sewer_network_V2, siblings inherit it.
# Add a network block only when intentionally changing network behavior.

### Ensemble execution helpers

The `sewertris.ensemble` module supports subprocess-based workers. This is useful because SWMM/PySWMM simulations are safer when isolated by process.

| Helper | Purpose |
|---|---|
| `run_project_sibling` | Clone a parent project and rerun a physical sibling, optionally stopping after a step. |
| `run_project_simulation` | Resume a prepared sibling at the SWMM/scenario stage. |
| `run_project_sibling_from_file` | Read a JSON payload and run the sibling or simulation worker. |

The example notebooks split ensembles into stages:

- `example_05A_Stillwater_prepare_ensemble_siblings.ipynb`: generate physical sibling projects through design.
- `example_05B_Stillwater_run_ensemble_simulations.ipynb`: run SWMM scenarios and save `flows.nc`.
- `example_05C_Stillwater_compare_ensemble_outputs.ipynb`: compare hydrographs, volumes, and peaks across ensembles.

In [8]:
# Example ensemble manifest pattern used by notebooks 05A-05C
from pathlib import Path

ENSEMBLE_ROOT = Path("Examples/output_example_5_ensembles")
N_REALIZATIONS = 3

sibling_specs = []
for i in range(1, N_REALIZATIONS + 1):
    sibling_specs.append(
        {
            "ensemble": "seed_only",
            "realization": i,
            "name": f"Stillwater seed-only realization {i:02d}",
            "output_dir": ENSEMBLE_ROOT / "seed_only" / f"run_{i:03d}",
            "changes": seed_only_changes(i),
        }
    )

sibling_specs[:1]

[{'ensemble': 'seed_only',
  'realization': 1,
  'name': 'Stillwater seed-only realization 01',
  'output_dir': PosixPath('examples/output_example_5_ensembles/seed_only/run_001'),
  'changes': {'layout_seed': 1001, 'rainfall': {'random_seed': 43}}}]

## 8. Flow-Component Experiments

SewerTris separates outlet flow into components using SWMM pollutant tags:

- **BWF/DWF**: dry-weather sanitary base flow.
- **GWI**: groundwater infiltration represented through baseline inflows.
- **RDII**: rainfall-derived inflow and infiltration, computed as the remaining wet-weather response after DWF and GWI separation.
- **Total flow**: modeled outlet link flow, usually at `P_OUTLET`.

Interesting ensembles usually vary one response mechanism at a time:

| Ensemble family | Parameters to vary | Expected hydrograph signal |
|---|---|---|
| `seed_only` | `layout_seed`, rainfall `random_seed` | Baseline stochastic spread. |
| `rainfall_intensity` | rainfall depth, storm probability, duration | RDII peak magnitude and timing. |
| `spatial_gwi` | GWI raster min/max and seed | Total baseline and GWI volume. |
| `spatial_rdii` | RDII density raster and impervious scaling | RDII volume and wet-weather peaks. |
| `drainage_shift` | topography/outlet direction/smoothing | Network slopes, timing, attenuation, routing response. |
| `urban_morphology` | tetromino set, layout seed, cell size | Pipe topology, lengths, accumulation, response spread. |

The cleanest analysis usually compares families rather than mixing many unrelated random changes into a single ensemble.

## 9. Plotting and Diagnostics

The plotting API is organized around three needs: inspect one model, compare two models, and compare ensembles.

| Function | Purpose |
|---|---|
| `plot_domain_mask` | Inspect the rasterized domain mask. |
| `plot_tetromino_set` | Show available tetromino pieces. |
| `plot_filled_board_shapefile` | View generated Tetris blocks. |
| `plot_roads` | View road centerlines/polygons. |
| `plot_blocks_landuse` | View land-use assignment. |
| `plot_dem_tif` | View synthetic or embedded DEM. |
| `plot_sewer_network_all` | Inspect main, secondary, tertiary, and unresolved pipes. |
| `plot_final_design_color_by_diameter` | Inspect designed pipe sizes. |
| `plot_flow_components_v2` | Plot one decomposed outlet hydrograph. |
| `plot_two_models` | Compare two projects across layout, roads, DEM, network, design, flows, and more. |
| `plot_ensemble_results` | Compare many sibling flow-component outputs by hydrograph, volume, and peak flow. |

`plot_ensemble_results` is designed for large ensembles. It shows all individual hydrographs as thin lines, group medians as heavier lines, and separate volume/peak panels for BWF/DWF, GWI, and RDII so that components with different scales remain readable.

In [9]:
# Plotting patterns
RUN_PLOTTING = False

if RUN_PLOTTING:
    base_project = st.SewerTrisProject.load("Examples/output_example_2_project/sewertris_project.json")
    sibling_project = st.SewerTrisProject.load("Examples/output_example_5_ensembles/seed_only/run_001/sewertris_project.json")

    st.plot_two_models(
        "flow_components",
        base_project,
        sibling_project,
        labels=("Base", "Sibling 001"),
        scenario_name="bwf_gwi_rdii",
    )

    simulation_manifest = pd.read_csv("Examples/output_example_5_ensembles/simulation_manifest.csv")
    simulation_manifest["base_model"] = "Stillwater V2 base"
    st.plot_ensemble_results(
        simulation_manifest,
        group_cols=("base_model", "ensemble"),
        title="Stillwater Ensemble Outlet Flow Components",
    )

## 10. Recommended Example Notebooks

| Notebook | Role |
|---|---|
| `example_01_my_first_sewertris_project.ipynb` | Minimal project-style workflow. |
| `example_02_Stillwater_sewertris_project.ipynb` | Full Stillwater project workflow with V2 network generation. |
| `example_03_Stillwater_4tetromino_sewertris_project.ipynb` | Variant layout using a smaller tetromino set. |
| `example_04_Stillwater_GWI_RDII_heterogeneity_sewertris_project.ipynb` | Spatially heterogeneous GWI/RDII workflow. |
| `example_05A_Stillwater_prepare_ensemble_siblings.ipynb` | Prepare physical siblings for ensemble analysis. |
| `example_05B_Stillwater_run_ensemble_simulations.ipynb` | Run SWMM simulations for prepared siblings. |
| `example_05C_Stillwater_compare_ensemble_outputs.ipynb` | Compare hydrographs, volumes, and peak flows across siblings. |

## 11. Documentation Maintenance Checklist

Use this checklist when the repository evolves:

- Keep installation commands synchronized with `environment_sewertris.yml` and `pyproject.toml`.
- Keep workflow step names synchronized with `SewerTrisProject.record_step` calls.
- Document new sibling change keys when they are added to the rerun inference map.
- Keep plotting function descriptions synchronized with `src/sewertris/plots.py`.
- Make sure example notebooks use the current recommended APIs, especially `complete_tetris_layout(seed=...)` and `generate_sewer_network_V2(...)`.
- Prefer project-based examples for anything intended to support siblings or ensembles.
- Keep old algorithm calls commented only where they are useful for historical reference.

A good future direction is to export this notebook into Markdown pages for the repository documentation site while keeping the notebook as the executable, readable source.